# 13 · Prueba multicriterio sobre la plantilla de base real — nivel EXPERTO

**Que hace**: corre el motor (`linkage()`) **dos veces sobre las mismas fuentes** — sin
variables extra (OFF) y con las que usted declare (ON: telefono, correo, direccion,
municipio, CIIU, sitio web...) — y muestra que cambia: grupos, precision / recall / F1
por pares, casos acertados y, sobre todo, **los pares concretos que se unen o se separan**
con los valores de las dos filas, para auditarlos a ojo.

**Estructura**:
1. **Que es una prueba multicriterio** y que comparadores existen (tabla generada desde
   el registro de la libreria, no copiada a mano).
2. **Demo autocontenida** sobre la plantilla de `data/ejemplos/` (536 registros inventados,
   278 grupos verdaderos, 24 casos): OFF contra ON, cifras, aporte de cada variable y pares
   que cambian.
3. **Sus datos**: lo mismo sobre su Excel con hojas `FUENTE_*` (y, si trae `VERDAD_GRUPOS`,
   `CASOS` y `ETIQUETAS_PARES`, la evaluacion; sin verdad, solo los pares que cambian).
4. **Exportacion**: un Excel con la correlativa ON, la correlativa OFF y la hoja de pares que
   cambian, mas el JSON de cifras, en `RUTA_RESULTADOS`.

**Para que tamano**: la plantilla y una **muestra** de su base (miles a decenas de miles de
registros; unos 15 s por 536 registros, porque corre el motor dos veces y guarda todos los pares
predichos en memoria). No es el camino para la base completa de 2-5 M de registros: para eso, una
sola configuracion en los notebooks 04 / 06.

La logica **no vive en las celdas**: `record_linkage.evaluation.multicriterio` (una sola
implementacion, con pruebas en `tests/test_multicriterio.py`; `scripts/multicriterio_plantilla.py`
corre doce configuraciones con ella). Lectura de los resultados y de lo que esta prueba NO
demuestra: [`docs/PLANTILLA_BASE_REAL.md`](../docs/PLANTILLA_BASE_REAL.md) §5.1 y §8.

> En Colab Free, suba su archivo a `/content/datos` y descargue `/content/resultados`
> antes de cerrar la sesion; el disco local es efimero.
>
> Motor: `rues-linker 0.22.4` + `evaluation.multicriterio` (entro despues del tag
> v0.22.4: la Celda A instala desde la rama `main` si el modulo no esta). **Hasta que el PR de
> la rama `claude/multicriterio-plantilla` se fusione, `main` no trae el modulo**: en Colab la
> Celda A se detiene y dice exactamente que cambiar (`ORIGEN_GIT` a esa rama). Cuando salga el
> siguiente tag se fijara el tag, como en los demas notebooks.

**Glosario** (lo que la salida nombra sin explicar):
- **motor** = `linkage()`; **scorer** = la fase que puntua cada pareja candidata (L3);
  **parquet de L1** = el archivo con la tabla limpia que el scorer lee; **SQLite** = la base de
  datos local con los intermedios (`RUTA_TRABAJO`); **checkout** = una copia del repositorio en disco.
- **GRUPOS** = empresas distintas que el motor cree ver; **PARES_PREDICHOS** = parejas de
  registros que une; **HUELLA** = identificador de la particion (misma huella = resultado identico).
- **regimen** = si el par tiene NIT en los dos registros (CON_ID), en ninguno (SIN_ID) o en uno
  (MIXTO); **singleton** = registro que debe quedar solo; **filtro de NIT distinto** = el motor no
  une dos NIT distintos aunque el nombre coincida; **`min_name_similarity`** = parecido minimo de
  nombre (0,30) para que un par siga vivo.
- **Wilson 95 %** = margen de error de la precision y el recall cuando hay pocos pares
  etiquetados; **formulas neutralizadas** = las celdas del Excel se escriben como texto, nunca
  como formula.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import importlib.util
import pathlib
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

VERSION_MINIMA = "0.22.4"
# El modulo evaluation.multicriterio entro DESPUES del tag v0.22.4: hasta el siguiente tag se
# instala desde la rama main del repositorio (version >= 0.22.4 con el modulo presente; main
# puede ir por delante del tag, por eso no se exige igualdad exacta). Hasta que el PR de la
# rama claude/multicriterio-plantilla se fusione, main NO trae el modulo: esta celda lo
# detecta y dice que cambiar (ORIGEN_GIT a esa rama).
ORIGEN_GIT = "rues-linker @ git+https://github.com/EnriqueForero/rues-linker_v2@main"
RAMA_CON_EL_MODULO = "claude/multicriterio-plantilla"
MODULO_NECESARIO = ("evaluation", "multicriterio", "__init__.py")  # es un paquete


def _version_ok(v: str, minima: str = VERSION_MINIMA) -> bool:
    """True si la version instalada es la minima o posterior."""

    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())

    try:
        return _t(v) >= _t(minima)
    except Exception:
        return False


def _modulo_presente() -> bool:
    """True si el paquete instalado trae evaluation/multicriterio/__init__.py (por archivo, sin importarlo)."""
    spec = importlib.util.find_spec("record_linkage")
    if spec is None or not spec.submodule_search_locations:
        return False
    raiz = pathlib.Path(next(iter(spec.submodule_search_locations)))
    return raiz.joinpath(*MODULO_NECESARIO).is_file()


def _instalar(argumentos: list[str]) -> None:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *argumentos, ORIGEN_GIT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            "  Verifique que Colab tenga salida a GitHub y vuelva a ejecutar esta celda.\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )


def preparar_entorno() -> None:
    """Garantiza rues-linker >= 0.22.4 CON el paquete evaluation/multicriterio/ (main hasta el siguiente tag)."""
    try:
        instalada = version("rues-linker")
    except PackageNotFoundError:
        instalada = None
    if instalada is not None and _version_ok(instalada) and _modulo_presente():
        import record_linkage

        if not _version_ok(record_linkage.__version__):
            raise RuntimeError(
                "Hay otra version cargada en memoria. Reinicie el entorno y reintente."
            )
        print(f"rues-linker {record_linkage.__version__} con evaluation.multicriterio — listo")
        return
    if instalada is None:
        print("rues-linker no instalado; instalando desde GitHub (main)...")
        _instalar([])
    elif not _modulo_presente():
        # pip NO reinstala una URL git con la misma version ya instalada (p. ej. la 0.22.4 del
        # tag que dejo otro notebook): sin --force-reinstall diria 'already satisfied' y el
        # modulo nunca llegaria. --no-deps: las dependencias ya estan y no se tocan.
        print(f"rues-linker {instalada} sin evaluation.multicriterio; reinstalando desde main...")
        _instalar(["--force-reinstall", "--no-deps"])
    else:
        print(f"rues-linker {instalada} es anterior a {VERSION_MINIMA}; actualizando...")
        _instalar(["--upgrade"])
    importlib.invalidate_caches()
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "La libreria se actualizo, pero una version anterior sigue cargada en memoria. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a ejecutar esta celda."
        )
    if not _modulo_presente():
        raise RuntimeError(
            f"Se instalo rues-linker desde '{ORIGEN_GIT.split('@')[-1]}', pero ese origen todavia "
            "no trae el paquete evaluation/multicriterio/ (el PR de la rama "
            f"{RAMA_CON_EL_MODULO} no se ha fusionado a main). Reiniciar el entorno NO lo arregla. "
            "Que hacer: espere al merge, o cambie ORIGEN_GIT a "
            f"'rues-linker @ git+https://github.com/EnriqueForero/rues-linker_v2@{RAMA_CON_EL_MODULO}' "
            "y vuelva a correr esta celda. Si pip dijo 'already satisfied', ejecute antes "
            "`pip uninstall -y rues-linker`."
        )
    import record_linkage

    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere {VERSION_MINIMA} o "
            "posterior. Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a "
            "correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} con evaluation.multicriterio — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A1 — RUTAS  (donde estan sus datos y donde quedan los resultados)
# ══════════════════════════════════════════════════════════════════════════
# En Colab Free suba sus archivos a /content/datos (este notebook no monta
# Drive) y descargue RUTA_RESULTADOS antes de cerrar la sesion: el disco local
# es efimero. RUTA_TRABAJO guarda los intermedios de linkage() (SQLite, parquet
# de L1) y es disco LOCAL: nunca Drive. tests/test_notebooks_contrato.py
# sobreescribe estas rutas para correr el notebook sobre data/ejemplos.
# La carpeta RUTA_DATOS la crea la primera celda EJECUTAR (seccion 1): tras correrla
# aparece 'datos' en el panel Archivos de Colab (icono de carpeta a la izquierda) y puede
# arrastrar alli su Excel. No se crea aqui porque la prueba del contrato inyecta sus rutas
# justo despues de esta celda y un mkdir de /content fallaria fuera de Colab.
from pathlib import Path

RUTA_DATOS = Path("/content/datos")  # carpeta con sus archivos
RUTA_RESULTADOS = Path("/content/resultados/13")  # carpeta de salida (descarguela)
RUTA_TRABAJO = Path("/content/rues_linker_work/13")  # intermedios: disco LOCAL, nunca Drive

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — LO UNICO QUE SE EDITA
# ══════════════════════════════════════════════════════════════════════════
# Su archivo, dentro de RUTA_DATOS: un Excel con hojas FUENTE_* (FUENTE_CRM,
# FUENTE_DIAN...; opcionales VERDAD_GRUPOS, CASOS y ETIQUETAS_PARES) o una carpeta
# con plantilla_fuente_*.csv. Formato exacto: docs/PLANTILLA_BASE_REAL.md §2.
ARCHIVO = "plantilla_base_real.xlsx"

# Las variables extra: columna CON EL NOMBRE DE SU ARCHIVO (MUNICIPIO, no CIUDAD),
# peso (el puntaje es 0,5·nombre + 0,5·nit + Σ peso·similitud, umbral 0,45) y tipo.
# Use tipos con FIRMADO = si en la tabla de la seccion 1 (email_signed, telefono_signed,
# tipo_direccion, categoria_tolerante_signed, conjunto_signed...): castigan la discrepancia
# y valen 0 si falta un lado. tipo_conjunto, tipo_geo y tipo_numerico NO castigan, y
# `categorical` premia el dato ausente. Pesos entre 0,05 y 0,10 por variable, suma <= 0,30:
# sobre la plantilla 0,10, 0,15 y 0,30 dieron exactamente lo mismo (subir el peso no es
# una prueba, es una apuesta). Columna -> tipo y peso sugerido: tabla de la seccion 1.
VARIABLES_EXTRA = [
    {"column": "EMAIL", "weight": 0.10, "type": "email_signed"},
    {"column": "TELEFONO", "weight": 0.10, "type": "telefono_signed"},
    # {"column": "MUNICIPIO", "weight": 0.05, "type": "categoria_tolerante_signed"},  # CALI = SANTIAGO DE CALI
    # {"column": "DIRECCION", "weight": 0.05, "type": "tipo_direccion"},  # tokens; premio 38 de 106 negativos
]

NOMBRE_CORRIDA = "multicriterio"  # prefijo de los archivos en RUTA_RESULTADOS

## 1 · Que es una prueba multicriterio y que comparadores existen

El motor decide con el **NIT** y la **razon social**. Una prueba multicriterio le da ademas
otras columnas como evidencia, cada una con un **peso** y un **comparador** (`type`):

```
puntaje = 0,5 · similitud(nombre) + 0,5 · similitud(nit) + Σ peso · similitud(variable)
se une si puntaje ≥ 0,45
```

- Los comparadores **firmados** son los que la tabla de abajo marca `FIRMADO = si`: todos los
  `*_signed` y los `tipo_*` **salvo `tipo_conjunto`, `tipo_geo` y `tipo_numerico`** (esos solo
  premian). Devuelven −1..+1: **premian** la coincidencia, **castigan** la discrepancia y valen 0
  si falta un lado. Son los que pueden separar dos empresas distintas. Para CIIU use
  `conjunto_signed`, no `tipo_conjunto`.
- Los **no firmados** (`categorical`, `token_set_ratio`, `exact_or_zero`, `tipo_conjunto`,
  `tipo_geo`, `tipo_numerico`) solo premian; `categorical` ademas devuelve 0,5 cuando falta un
  lado (premia el dato ausente).
- En los pares **sin NIT** el puntaje de NIT es cero: las variables extra son las que pueden
  subirlos o bajarlos del umbral. Pero se suman **despues** del filtro de nombre
  (`min_name_similarity`) y del filtro de NIT distinto: no rescatan un par que esos
  filtros descartan.

Dos reglas medidas (CLAUDE.md §5): `TELEFONO` y `EMAIL` con el comparador exacto no sirven
(el mismo telefono viaja en tres formatos); use `telefono_signed` y `email_signed`. Y una
lista (CIIU) se compara por solapamiento (`conjunto_signed`), no por igualdad.

**Que tipo y que peso para cada columna de la plantilla** (medido sobre `data/ejemplos`,
`docs/PLANTILLA_BASE_REAL.md` §5.1; en su base, mida):

| Columna | Tipo | Peso sugerido | Que hace, en llano |
|---|---|---|---|
| `TELEFONO` | `telefono_signed` | 0,10 | compara los ultimos 7 digitos: el mismo numero en tres formatos cuenta como igual; numero distinto castiga |
| `EMAIL` | `email_signed` | 0,10 | dominio distinto castiga; mismo dominio, parecido de la parte local |
| `DIRECCION` | `tipo_direccion` | 0,05 | tokens de la direccion (`Tv 145 # 8-72` y `TRANSVERSAL 145 No. 8 - 72` cuentan como iguales); en la plantilla reconocio 147 de 147 positivos pero premio 38 de 106 negativos |
| `MUNICIPIO`, `DEPARTAMENTO` | `categoria_tolerante_signed` | 0,05 | `CALI` y `SANTIAGO DE CALI` cuentan como iguales; coincidir de ciudad es evidencia debil (premio 21 de 172 negativos) |
| `CIIU` | `conjunto_signed` | 0,05 | solapamiento de listas de codigos (comparten alguno), no igualdad |
| `SITIO_WEB` | `exact_signed` | 0,05 | igualdad exacta; en la plantilla casi no hay con que medirlo (8 pares) |

**Regla de pesos**: entre 0,05 y 0,10 por variable, suma ≤ 0,30. Sobre la plantilla 0,10, 0,15 y
0,30 dieron **exactamente lo mismo** (§5.1, configuraciones D y L): subir el peso no es una prueba,
es una apuesta. Cada cambio en la Celda B = volver a correr y volver a leer `PARES_QUE_CAMBIAN`.


In [ ]:
# ══ EJECUTAR · los comparadores registrados en la libreria ══════════════════
import pandas as pd
from IPython.display import display

from record_linkage.evaluation.multicriterio import (
    LEYENDA_RESUMEN,
    comparar_off_on,
    describir_libro,
    descripcion_tipos,
    exportar_comparacion,
    leer_etiquetas_de,
    localizar_plantilla,
)
from record_linkage.evaluation.plantilla_base_real import leer_plantilla

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

# Crea la carpeta de datos: en Colab aparece 'datos' en el panel Archivos (icono de carpeta
# a la izquierda) y puede arrastrar alli su Excel antes de la seccion 3.
RUTA_DATOS.mkdir(parents=True, exist_ok=True)


def mostrar_comparacion(comparacion) -> None:
    """Lo que una persona mira de una comparacion OFF/ON (solo presentacion: la logica vive en la
    libreria). Se usa en la demo (seccion 2) y sobre sus datos (seccion 3)."""
    print("\nResumen (pares contra VERDAD_GRUPOS; ETIQUETAS_* contra ETIQUETAS_PARES):")
    display(comparacion.resumen)
    print(LEYENDA_RESUMEN)
    print(
        "Columnas extra que vio el motor (True = llego al scorer; si alguna no llega, la "
        f"libreria se detiene antes): {comparacion.con.columnas_extra_en_scorer}"
    )
    if comparacion.con.tabla_etiquetas is not None:
        print(
            "\nContra ETIQUETAS_PARES por regimen, con intervalo de Wilson 95 % (configuracion ON):"
        )
        display(comparacion.con.tabla_etiquetas)
    c = comparacion.con.cambios_vs_base
    print(
        f"\nCasos que mejoran con las extras: {c['casos_que_mejoran']} · que empeoran: "
        f"{c['casos_que_empeoran']} · pares que se unen: {len(c['pares_ganados'])} · "
        f"que se separan: {len(c['pares_perdidos'])}"
    )


TIPOS = descripcion_tipos()
print(f"{len(TIPOS)} comparadores registrados; use el nombre de TIPO en VARIABLES_EXTRA")
display(TIPOS)

## 2 · Demo autocontenida: OFF contra ON sobre la plantilla

La plantilla de `data/ejemplos/` (empresas inventadas: KRONOS, ZELVORA, TRIVENTA...) trae
tres fuentes (CRM 187, RUES 176, ADUANAS 173 registros), `VERDAD_GRUPOS` (278 grupos),
`CASOS` (24 casos: NIT con y sin DV, erratas, siglas, sucursales, homonimos...) y
`ETIQUETAS_PARES` (109 pares etiquetados, **106 evaluables**: los 3 `UNCLEAR` se descartan al
leer). Si `RUTA_DATOS` no la trae, el notebook la busca en un checkout del repositorio y, si
tampoco, la **descarga de GitHub (`main`)**.

Medido el 2026-10-14 (`docs/PLANTILLA_BASE_REAL.md` §5.1): sin extras, 281 grupos,
P 0,9875 / R 0,9754, 18 de 24 casos; con correo + telefono firmados a 0,10, 283 grupos,
**P 0,9937** / R 0,9754, 20 de 24: separa los dos falsos positivos de borde (el singleton
`OLVERANA`/`LAVIRENA` por dominio de correo distinto y el ambiguo `JUAN CARLOS PEREZ GOMEZ`
por telefono distinto) y no mueve ningun par verdadero. **Son datos inventados**: estas cifras
muestran que la fontaneria funciona, no lo que pasara con su base (ver *Interpretacion y limites*
al final). Compruebelo aqui y cambie `VARIABLES_EXTRA` en la Celda B para ver que hace cada
variable.


In [ ]:
# ══ EJECUTAR · demo: el motor sin extras y con VARIABLES_EXTRA sobre la plantilla ══
# Donde buscarla, en orden: una descarga previa, RUTA_DATOS, un checkout del repositorio;
# si no esta en ninguna, se descargan los CSV de data/ejemplos (rama main) a plantilla_demo/.
RUTA_PLANTILLA = localizar_plantilla(
    [
        RUTA_DATOS / "plantilla_demo",
        RUTA_DATOS,
        Path.cwd() / "data" / "ejemplos",
        Path.cwd().parent / "data" / "ejemplos",
    ],
    descargar_a=RUTA_DATOS / "plantilla_demo",
)
print(f"Plantilla: {RUTA_PLANTILLA}")
LIBRO_DEMO = leer_plantilla(RUTA_PLANTILLA)
ETIQUETAS_DEMO = leer_etiquetas_de(RUTA_PLANTILLA)
print(describir_libro(LIBRO_DEMO, ETIQUETAS_DEMO))
DEMO = comparar_off_on(
    LIBRO_DEMO, ETIQUETAS_DEMO, VARIABLES_EXTRA, RUTA_TRABAJO / "demo", nombre=NOMBRE_CORRIDA
)
mostrar_comparacion(DEMO)

In [ ]:
# ══ EJECUTAR · aporte de cada variable con su comparador ════════════════════
# Sobre los 325 pares positivos (mismo grupo) y 210 negativos dificiles: cuantos pares con
# las dos celdas llenas PREMIA y cuantos CASTIGA cada comparador. SEPARACION_PP = % de
# positivos premiados menos % de negativos premiados: lo que separa, no la tasa de llenado.
# Una variable que premia a los dos lados por igual no discrimina, por alta que sea su
# tasa de coincidencia. *_PREMIA_UN_LADO_VACIO = pares que el comparador premia con una
# celda vacia: 0 con los firmados; con `categorical` es la trampa del dato ausente.
display(DEMO.aporte)

In [ ]:
# ══ EJECUTAR · los pares que cambian de decision, con sus valores (auditoria a ojo) ══
# CAMBIO = "se une" (ON los junta y OFF no) o "se separa" (OFF los juntaba y ON no);
# VERDADERO = si segun VERDAD_GRUPOS son la misma empresa. Las columnas _1 y _2 son
# las dos filas tal cual vienen en las fuentes.
print(f"{len(DEMO.pares_que_cambian)} pares cambian de decision al activar las extras")
display(DEMO.pares_que_cambian)
for ruta in exportar_comparacion(DEMO, RUTA_RESULTADOS, f"demo_plantilla_{NOMBRE_CORRIDA}"):
    print(f"  Escrito: {ruta}")

## 3 · Sus datos

Suba a `RUTA_DATOS` su Excel con hojas `FUENTE_*` (una por fuente; `docs/PLANTILLA_BASE_REAL.md`
§2 y §3). **Minimo por hoja**: `ID_REGISTRO_FUENTE` (**unico en todas las hojas**: prefijo de la
fuente + su clave, `CRM-000017`; si cada hoja va 1..N la lectura se detiene y lo dice),
`RAZON_SOCIAL`, `NIT` (vacio si no hay) y `MUNICIPIO`; si falta `NIT` o `MUNICIPIO` la lectura
avisa (el motor correria solo por nombre, o sin ciudad). Las demas columnas de la plantilla
(`TELEFONO`, `EMAIL`, `DIRECCION`, `DEPARTAMENTO`, `CIIU`, `SITIO_WEB`...) son opcionales y solo
cuentan si las declara en `VARIABLES_EXTRA`.

Que cifras va a ver, segun lo que traiga el libro (la celda lo imprime antes de correr):
- `VERDAD_GRUPOS` con `ID_REGISTRO_FUENTE` e `ID_GRUPO` → precision, recall y F1 por pares; si
  ademas trae la columna `CASO` y la hoja `CASOS` → la tabla de casos y el aporte por variable;
- `ETIQUETAS_PARES` → P y R globales en el resumen, y por regimen con intervalos de Wilson 95 %
  en la tabla siguiente (y en el JSON, clave `etiquetas_pares`);
- sin verdad, el notebook igual corre OFF y ON y le entrega **los pares que cambian de decision**
  con sus valores: eso es lo que una persona revisa.

Si falta el archivo, la celda se detiene con un mensaje que dice que hacer (no se inventa nada).


In [ ]:
# ══ EJECUTAR · OFF contra ON sobre su archivo ═══════════════════════════════
RUTA_MIS_DATOS = RUTA_DATOS / ARCHIVO
if not RUTA_MIS_DATOS.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_MIS_DATOS}. Que hacer: suba su Excel con hojas FUENTE_* (o la "
        f"carpeta con plantilla_fuente_*.csv) a {RUTA_DATOS} y escriba su nombre en ARCHIVO "
        "(Celda B). La seccion 2 ya corrio sobre la plantilla de demostracion."
    )
MIS_DATOS = leer_plantilla(RUTA_MIS_DATOS)
MIS_ETIQUETAS = leer_etiquetas_de(RUTA_MIS_DATOS)
print(describir_libro(MIS_DATOS, MIS_ETIQUETAS))
RESULTADO = comparar_off_on(
    MIS_DATOS, MIS_ETIQUETAS, VARIABLES_EXTRA, RUTA_TRABAJO / "sus_datos", nombre=NOMBRE_CORRIDA
)
mostrar_comparacion(RESULTADO)
if RESULTADO.aporte is not None:
    print("\nAporte de cada variable (lectura: seccion 2):")
    display(RESULTADO.aporte)
print(f"\n{len(RESULTADO.pares_que_cambian)} pares cambian de decision al activar las extras")
display(RESULTADO.pares_que_cambian)

## 4 · Exportar

Un Excel `<NOMBRE_CORRIDA>.xlsx` con las hojas `RESUMEN`, `PARES_QUE_CAMBIAN`,
`CORRELATIVA_ON`, `CORRELATIVA_OFF` y, con verdad, `CASOS` (acerto sin y con extras, caso
por caso) y `APORTE`; mas `<NOMBRE_CORRIDA>.json` con las cifras, los ids de los pares
que cambian, los FP y FN y la configuracion exacta que recibio el motor (ningun dato de
registro). Se escriben con las primitivas del estandar de salida (formulas neutralizadas).


In [ ]:
# ══ EJECUTAR · Excel + JSON en RUTA_RESULTADOS ═══════════════════════════════
if "RESULTADO" not in globals():
    raise RuntimeError(
        "No hay resultado de la seccion 3 (su archivo). La demo ya quedo exportada en "
        f"{RUTA_RESULTADOS} como demo_plantilla_{NOMBRE_CORRIDA}.xlsx/.json; para exportar los "
        f"suyos, suba su archivo a {RUTA_DATOS}, escriba su nombre en ARCHIVO (Celda B) y corra "
        "la seccion 3."
    )
for ruta in exportar_comparacion(RESULTADO, RUTA_RESULTADOS, NOMBRE_CORRIDA):
    print(f"  Escrito: {ruta}")
print("Descargue RUTA_RESULTADOS antes de cerrar la sesion: el disco de Colab es efimero.")

### Interpretacion y limites

- **Regla de decision del proyecto**: «unir mal duele mas que dejar sin unir». Entre dos
  configuraciones, gana la de mayor **precision** por pares; a igual precision, mayor recall;
  a igual recall, la mas simple. Con esa regla, sobre la plantilla la recomendada es
  **correo + telefono firmados a 0,10** (la Celda B trae exactamente eso).
- **Lea los pares que cambian, no solo las cifras.** Con 325 pares verdaderos, un par mal
  unido mueve la precision en la tercera cifra decimal; en produccion es una fusion
  escandalosa. La hoja `PARES_QUE_CAMBIAN` existe para eso.
- **Que hacer con cada fila de `PARES_QUE_CAMBIAN`** (`VERDADERO` sale de `VERDAD_GRUPOS`; en
  su base sin verdad va vacio y decide usted a ojo con las columnas `_1` / `_2`):
  - `se une` + `VERDADERO = no` (o vacio y a ojo son dos empresas): la variable **premia de
    mas**. Quitela o baje su peso; **nunca suba el umbral** para compensar.
  - `se separa` + `VERDADERO = si` (o a ojo son la misma: una sucursal con otro telefono): la
    variable **castiga un par verdadero**. Baje su peso o use una que no cambie entre sedes.
  - `se une` + `si`, o `se separa` + `no`: la variable **ayuda**; anotelo con su cifra y sus pares.
  - **0 pares cambian**: con ese peso la variable no decide nada en su base. No la suba «a ver si
    hace algo»: en la plantilla 0,10, 0,15 y 0,30 dieron lo mismo.
  - Cada cambio en la Celda B = volver a correr y volver a leer `PARES_QUE_CAMBIAN`; lo que se
    acepta se escribe con su cifra y sus pares (la hoja `PARES_QUE_CAMBIAN` del Excel es el registro).
- **Lo que la plantilla NO prueba**: es sintetica. Los correos de un mismo grupo son
  identicos y ningun negativo comparte correo ni telefono, asi que la precision del contacto
  sale perfecta por construccion; ningun falso negativo es rescatable por las extras (los
  filtros de nombre y de NIT van antes); y 0,10 o 0,15 de peso dan lo mismo. El riesgo real
  del contacto (el contador que lleva veinte empresas, el telefono del centro comercial) se
  mide con **su** base y **sus** etiquetas, no aqui.
- **Hallazgos del motor que hay que conocer** (`docs/PLANTILLA_BASE_REAL.md` §5.1):
  `MUNICIPIO` se declara con ese nombre y la libreria lo traduce a `CIUDAD` (el motor
  renombra la columna de ciudad antes de validar las extras); las extras se suman despues de
  los filtros de nombre y de NIT; `categorical` premia el dato ausente; las variables extra
  son dicts `{"column", "weight", "type"}`, nunca tuplas.
- Para medir **doce configuraciones** de una vez, con determinismo en subprocesos:
  `python scripts/multicriterio_plantilla.py --datos data/ejemplos --salida mi_medicion.json`.
